# 06 · Selección por validación, test una sola vez, tabla 3×3 y `predict_raw`

1. **Selección (solo validación):** para cada (entrada, arquitectura) se elige la corrida con menor `val_loss` (MLP: con y sin Dropout).
2. **Test, una sola vez:** cada modelo elegido se evalúa sobre los 305 pacientes de prueba. La celda se protege con `artifacts/test_results.json`: si existe, se lee en lugar de recalcular.
3. **Métricas:** por paciente (MAE, ME, DE, promediadas sobre pacientes — definición del anteproyecto) y agrupadas sobre todos los segmentos. La DE por paciente descuenta el sesgo de cada paciente; la **comparación con AAMI (ME ≤ 5, DE ≤ 8 mmHg) se hace con las agrupadas**.
**Nota de precisión numérica:** TimesFM se entrenó con autocast bf16, pero la inferencia (evaluación en test y `predict_raw`) se hace en **fp32**: en bf16 el mismo modelo da hasta 0.19 mmHg de diferencia según el tamaño de lote y hasta ≈ 1 mmHg frente a fp32, lo que impide reproducir las predicciones. La primera corrida de este notebook evaluó TimesFM en bf16 (`artifacts/test_results_bf16_primera_corrida.json`); se repitió en fp32 por reproducibilidad, sin cambiar la selección (hecha en validación). MLP y GRU no se ven afectados.
4. **Bundle** del mejor modelo (elegido por validación) y verificación de `predict_raw` contra las predicciones de test.

In [1]:
import sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "bp").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, torch
print("ROOT:", ROOT, "| cuda:", torch.cuda.is_available())

ROOT: C:\Users\USUARIO\Documents\Source\dl_bp_mimic_proyecto_integrador | cuda: True


In [2]:
import pandas as pd
from bp.experiment import get_data, load_label_stats, load_val_results, device
from bp.train import build_model, load_state, predict_norm
from bp.metrics import per_patient_metrics
from pathlib import Path
from bp.paths import ARTIFACTS, DATA_CARD
y_mean, y_std = load_label_stats()
dev = device()
vals = load_val_results()
print(len(vals), "corridas de validación:", sorted(vals))

15 corridas de validación: ['gru_ecg', 'gru_ppg', 'gru_ppg+ecg', 'mlp_do0.2_ecg', 'mlp_do0.2_ppg', 'mlp_do0.2_ppg+ecg', 'mlp_ecg', 'mlp_ppg', 'mlp_ppg+ecg', 'timesfm_lora_ecg', 'timesfm_lora_ppg', 'timesfm_lora_ppg+ecg', 'timesfm_probe_ecg', 'timesfm_probe_ppg', 'timesfm_probe_ppg+ecg']


## 1. Selección por `val_loss`

In [3]:
def family(r):
    if r['builder'] == 'build_timesfm': return 'timesfm_lora'
    if r['builder'] == 'build_probe_head': return 'timesfm_probe'
    return r['builder'].replace('build_', '')

best = {}
for name, r in vals.items():
    key = (family(r), '+'.join(r['inputs']))
    if key not in best or r['best_val_loss'] < best[key]['best_val_loss']:
        best[key] = r
# TimesFM es una estrategia de dos pasos (probe -> LoRA): en cada entrada se reporta el paso con menor val_loss
STEP = {'timesfm_lora': 'LoRA', 'timesfm_probe': 'probe'}
chosen = {inp: min(('timesfm_lora', 'timesfm_probe'), key=lambda f: best[(f, inp)]['best_val_loss']) for inp in ('ppg', 'ecg', 'ppg+ecg')}
print('paso de TimesFM elegido por validación:', {i: STEP[f] for i, f in chosen.items()})
sel = pd.DataFrame([{'modelo': k[0], 'entrada': k[1], 'corrida': r['run_name'], 'mejor_epoca': r['best_epoch'],
                     'val_loss': round(r['best_val_loss'], 4), 'val_MAE_PAS': round(r['val_mae_pas'], 2), 'val_MAE_PAD': round(r['val_mae_pad'], 2)}
                    for k, r in sorted(best.items())])
sel

paso de TimesFM elegido por validación: {'ppg': 'LoRA', 'ecg': 'probe', 'ppg+ecg': 'LoRA'}


,modelo,entrada,corrida,mejor_epoca,val_loss,val_MAE_PAS,val_MAE_PAD
0,gru,ecg,gru_ecg,1,0.9090,14.01,9.33
1,gru,ppg,gru_ppg,13,0.7973,13.20,8.52
2,gru,ppg+ecg,gru_ppg+ecg,5,0.8609,13.32,8.82
3,mlp,ecg,mlp_do0.2_ecg,1,0.9371,14.24,9.56
4,mlp,ppg,mlp_do0.2_ppg,6,0.9005,13.99,9.20
5,mlp,ppg+ecg,mlp_do0.2_ppg+ecg,0,0.9157,14.10,9.28
6,timesfm_lora,ecg,timesfm_lora_ecg,0,0.9102,14.12,8.97
7,timesfm_lora,ppg,timesfm_lora_ppg,2,0.7766,12.97,8.32
8,timesfm_lora,ppg+ecg,timesfm_lora_ppg+ecg,1,0.7883,13.20,8.33
9,timesfm_probe,ecg,timesfm_probe_ecg,7,0.8923,14.08,9.02


## 2. Evaluación en test (una sola vez)

In [4]:
TEST_FILE = ARTIFACTS / 'test_results.json'
test_cache = {}   # predicciones por segmento, solo para verificar predict_raw

def load_model(r):
    ck = torch.load(r['checkpoint'], weights_only=False)
    if ck['builder'] == 'build_probe_head':
        m = build_model('build_timesfm', len(ck['inputs']), 0, mode='probe')
    else:
        m = build_model(ck['builder'], len(ck['inputs']), 0, **ck['model_kw'])
    load_state(m, ck['state'])
    return m.to(dev).eval(), ck

def eval_test(r):
    inputs = r['inputs']
    if r['builder'] == 'build_probe_head':      # cabeza sobre embeddings de test precalculados
        te = get_data('test', inputs, embeddings=True)
        ck = torch.load(r['checkpoint'], weights_only=False)
        head = build_model('build_probe_head', te['X'].shape[1], 0).to(dev)
        load_state(head, ck['state'])
        pn = predict_norm(head, te['X'], 4096)
    else:
        te = get_data('test', inputs)
        m, ck = load_model(r)
        pn = predict_norm(m, te['X'], 32 if r['builder'] == 'build_timesfm' else 1024, amp=False)   # inferencia siempre en fp32
        del m
    pred = pn.cpu().numpy() * y_std + y_mean
    return pred, te['y'], te['patient']

if TEST_FILE.exists():
    test_res = json.loads(TEST_FILE.read_text(encoding='utf-8'))
    print("test_results.json ya existe: se lee, NO se vuelve a evaluar test")
else:
    test_res = {}
    # LB-0
    te = get_data('test', ['ppg'], to_device=False)
    test_res['lb0|-'] = per_patient_metrics(np.tile(y_mean, (len(te['y']), 1)), te['y'], te['patient'])
    for (fam, inp), r in sorted(best.items()):
        pred, y, pat = eval_test(r)
        test_res[f'{fam}|{inp}'] = {**per_patient_metrics(pred, y, pat), 'run_name': r['run_name']}
        test_cache[f'{fam}|{inp}'] = pred
        print(f'{fam:14s} {inp:8s}', {k: round(v, 2) for k, v in test_res[f'{fam}|{inp}'].items() if k in ('mae_pas', 'mae_pad')})
        torch.cuda.empty_cache()
    TEST_FILE.write_text(json.dumps(test_res, indent=1), encoding='utf-8')
print(len(test_res), "filas de test")

test_results.json ya existe: se lee, NO se vuelve a evaluar test
13 filas de test


## 3. Tabla final 3×3 (test, 305 pacientes)
Por paciente: MAE / ME / DE promediados sobre pacientes. Agrupado: ME y DE sobre todos los segmentos (comparables con AAMI).

In [5]:
NAMES = {'lb0': 'Línea Base (Media)', 'mlp': 'MLP', 'gru': 'GRU', 'timesfm': 'TimesFM 2.5'}
INP = {'ppg': 'PPG', 'ecg': 'ECG', 'ppg+ecg': 'PPG + ECG', '-': '—'}
rows = []
for key, m in test_res.items():
    fam, inp = key.split('|')
    is_tfm = fam.startswith('timesfm')
    rows.append({'Entrada': INP[inp], 'Modelo': NAMES['timesfm' if is_tfm else fam], 'Paso': STEP.get(fam, ''),
                 'seleccionado': (not is_tfm) or chosen[inp] == fam,
                 'val_loss': best[(fam, inp)]['best_val_loss'] if key != 'lb0|-' else np.nan,
                 'MAE PAS': m['mae_pas'], 'ME PAS': m['me_pas'], 'DE PAS': m['de_pas'],
                 'MAE PAD': m['mae_pad'], 'ME PAD': m['me_pad'], 'DE PAD': m['de_pad'],
                 'ME agr. PAS': m['pooled_me_pas'], 'DE agr. PAS': m['pooled_sd_pas'], 'ME agr. PAD': m['pooled_me_pad'], 'DE agr. PAD': m['pooled_sd_pad']})
full = pd.DataFrame(rows)
order_in = {'—': 0, 'PPG': 1, 'ECG': 2, 'PPG + ECG': 3}; order_m = {v: i for i, v in enumerate(NAMES.values())}
full = full.sort_values(by=['Entrada', 'Modelo', 'Paso'], key=lambda c: c.map(order_in) if c.name == 'Entrada' else (c.map(order_m) if c.name == 'Modelo' else c)).reset_index(drop=True)
full['AAMI agr. PAS'] = (full['ME agr. PAS'].abs() <= 5) & (full['DE agr. PAS'] <= 8)
full['AAMI agr. PAD'] = (full['ME agr. PAD'].abs() <= 5) & (full['DE agr. PAD'] <= 8)
df = full[full['seleccionado']].drop(columns=['seleccionado', 'val_loss']).reset_index(drop=True)      # tabla 3x3 principal
supp = full[full['Modelo'] == 'TimesFM 2.5'].drop(columns=['seleccionado']).reset_index(drop=True)       # ambos pasos de TimesFM
print(df.round(2).to_string(index=False)); print(); print(supp.round(3).to_string(index=False))
(ARTIFACTS / 'tabla_3x3.md').write_text(df.round(2).to_markdown(index=False), encoding='utf-8')
(ARTIFACTS / 'tabla_timesfm_ambos_pasos.md').write_text(supp.round(3).to_markdown(index=False), encoding='utf-8')
df.to_csv(ARTIFACTS / 'tabla_3x3.csv', index=False, encoding='utf-8')
df.round(2)

  Entrada             Modelo  Paso  MAE PAS  ME PAS  DE PAS  MAE PAD  ME PAD  DE PAD  ME agr. PAS  DE agr. PAS  ME agr. PAD  DE agr. PAD  AAMI agr. PAS  AAMI agr. PAD
        — Línea Base (Media)          13.82   -0.15   11.86     9.28    0.47    6.58        -0.14        17.28         0.47        11.66          False          False
      PPG                MLP          13.76    0.29   12.26     8.99    0.84    6.94         0.30        17.24         0.84        11.28          False          False
      PPG                GRU          13.10    0.49   11.65     8.20    1.11    6.55         0.50        16.35         1.11        10.45          False          False
      PPG        TimesFM 2.5  LoRA    12.70    0.13   11.47     7.92    0.36    6.52         0.14        15.97         0.36        10.22          False          False
      ECG                MLP          13.87   -0.52   12.10     9.24    0.27    6.95        -0.51        17.37         0.27        11.67          False          Fals

,Entrada,Modelo,Paso,MAE PAS,ME PAS,DE PAS,MAE PAD,ME PAD,DE PAD,ME agr. PAS,DE agr. PAS,ME agr. PAD,DE agr. PAD,AAMI agr. PAS,AAMI agr. PAD
0,—,Línea Base (Media),,13.82,-0.15,11.86,9.28,0.47,6.58,-0.14,17.28,0.47,11.66,False,False
1,PPG,MLP,,13.76,0.29,12.26,8.99,0.84,6.94,0.30,17.24,0.84,11.28,False,False
2,PPG,GRU,,13.10,0.49,11.65,8.20,1.11,6.55,0.50,16.35,1.11,10.45,False,False
3,PPG,TimesFM 2.5,LoRA,12.70,0.13,11.47,7.92,0.36,6.52,0.14,15.97,0.36,10.22,False,False
4,ECG,MLP,,13.87,-0.52,12.10,9.24,0.27,6.95,-0.51,17.37,0.27,11.67,False,False
5,ECG,GRU,,13.74,-0.52,11.85,8.99,-0.37,6.64,-0.52,17.18,-0.37,11.43,False,False
6,ECG,TimesFM 2.5,probe,13.66,-0.07,11.93,8.75,0.50,6.75,-0.06,17.03,0.50,11.04,False,False
7,PPG + ECG,MLP,,13.74,-0.86,12.00,8.93,-0.17,6.78,-0.85,17.23,-0.18,11.34,False,False
8,PPG + ECG,GRU,,13.27,-1.61,11.64,8.29,-0.82,6.69,-1.61,16.61,-0.82,10.72,False,False
9,PPG + ECG,TimesFM 2.5,LoRA,12.72,1.38,11.56,7.97,0.01,6.60,1.39,15.94,0.01,10.28,False,False


In [6]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
main = df[df['Modelo'].isin(['MLP', 'GRU', 'TimesFM 2.5'])]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for a, t in zip(ax, ['PAS', 'PAD']):
    piv = main.pivot(index='Entrada', columns='Modelo', values=f'MAE {t}')[['MLP', 'GRU', 'TimesFM 2.5']].loc[['PPG', 'ECG', 'PPG + ECG']]
    piv.plot.bar(ax=a, rot=0); a.set_ylabel('MAE por paciente (mmHg)'); a.set_title(f'MAE {t} en test')
    a.axhline(float(df.loc[df['Modelo'] == 'Línea Base (Media)', f'MAE {t}'].iloc[0]), color='k', ls='--', label='Línea base')
    a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(ARTIFACTS / 'fig_tabla_3x3.png', dpi=110)

## 4. Bundle del mejor modelo (elegido por **validación**, no por test) y `predict_raw`

In [7]:
from bp.predict import make_bundle, save_bundle, load_bundle, predict_raw, BUNDLE_PATH
from bp.contract import ContractError
primary = {k: r for k, r in best.items() if k[0] in ('mlp', 'gru') or chosen[k[1]] == k[0]}   # las 9 celdas de la tabla principal
(fam, inp), champ = min(primary.items(), key=lambda kv: kv[1]['best_val_loss'])
ck = torch.load(champ['checkpoint'], weights_only=False)
summary = {k: v for k, v in test_res[f'{fam}|{inp}'].items() if isinstance(v, float)}
extra = {}
model_kw = dict(ck['model_kw'])
if ck['builder'] == 'build_timesfm':   # los pesos base no viajan en el bundle: se fija su revisión exacta
    import huggingface_hub, transformers, peft
    from bp.models import TIMESFM_ID
    sha = Path(huggingface_hub.snapshot_download(TIMESFM_ID, local_files_only=True)).name
    model_kw['revision'] = sha
    extra = {'base_model': TIMESFM_ID, 'base_revision': sha,
             'versions': {'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__}}
bundle = make_bundle(ck['state'], ck['builder'], ck['inputs'], model_kw, y_mean, y_std,
                     json.loads(DATA_CARD.read_text(encoding='utf-8')), ck['best_epoch'], summary, amp=False, extra=extra)
save_bundle(bundle)
print("campeón por validación:", fam, inp, '| val_loss', round(champ['best_val_loss'], 4), '| bundle', BUNDLE_PATH.stat().st_size // 1024, 'KB')

campeón por validación: timesfm_lora ppg | val_loss 0.7766 | bundle 6470 KB


In [8]:
# Verificación: predict_raw sobre las señales CRUDAS de los pacientes de test == predicciones del pipeline de evaluación
from bp.split import load_split
from bp.dataset import split_rows
from bp.paths import DATA
bundle = load_bundle()
ids = load_split()['test']
rows, idx = split_rows(ids)
subjects = json.loads((ROOT / 'cache' / 'preprocessed' / 'subjects.json').read_text(encoding='utf-8'))
raw = {m: np.concatenate([np.load(DATA / m / f'{subjects[k]}_{m}.npy')[idx['seg_idx'][rows][idx['subject_idx'][rows] == k]]
                          for k in np.unique(idx['subject_idx'][rows])]) for m in ('ppg', 'ecg')}
# orden de filas: por paciente ascendente = orden de `rows`
assert len(raw['ppg']) == len(rows)
pred_raw = predict_raw(raw['ppg'] if 'ppg' in bundle['inputs'] else None, raw['ecg'] if 'ecg' in bundle['inputs'] else None, bundle=bundle)
pred_eval = test_cache[f'{fam}|{inp}'] if test_cache else eval_test(champ)[0]   # mismas predicciones que la tabla
y_te = np.load(ROOT / 'cache' / 'preprocessed' / 'labels.npy')[rows]
diff = np.abs(pred_raw - pred_eval)
tol = 1e-3
print(f"max |predict_raw - evaluación| = {diff.max():.2e} mmHg | fracción dentro de {tol} mmHg: {(diff <= tol).mean() * 100:.3f}% | "
      f"MAE de predict_raw vs etiquetas (mmHg): {np.abs(pred_raw - y_te).mean(0).round(2)}")
assert (diff <= tol).mean() > 0.9999

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

[transformers] TimesFm2_5Model LOAD REPORT from: google/timesfm-2.5-200m-transformers
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
output_projection_point.output_layer.weight       | UNEXPECTED |  | 
output_projection_point.residual_layer.weight     | UNEXPECTED |  | 
output_projection_quantiles.output_layer.weight   | UNEXPECTED |  | 
output_projection_quantiles.residual_layer.weight | UNEXPECTED |  | 
output_projection_quantiles.input_layer.weight    | UNEXPECTED |  | 
output_projection_point.input_layer.weight        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

[transformers] TimesFm2_5Model LOAD REPORT from: google/timesfm-2.5-200m-transformers
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
output_projection_point.output_layer.weight       | UNEXPECTED |  | 
output_projection_point.residual_layer.weight     | UNEXPECTED |  | 
output_projection_quantiles.output_layer.weight   | UNEXPECTED |  | 
output_projection_quantiles.residual_layer.weight | UNEXPECTED |  | 
output_projection_quantiles.input_layer.weight    | UNEXPECTED |  | 
output_projection_point.input_layer.weight        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


max |predict_raw - evaluación| = 4.96e-05 mmHg | fracción dentro de 0.001 mmHg: 100.000% | MAE de predict_raw vs etiquetas (mmHg): [12.7   7.92]


In [9]:
# Señales corruptas: deben fallar con una excepción descriptiva
ok_p, ok_e = raw['ppg'][:2], raw['ecg'][:2]
def must_fail(label, **kw):
    try:
        predict_raw(bundle=bundle, **kw)
    except ContractError as e:
        print(f'OK  {label}: {e}')
    else:
        raise AssertionError(f'{label}: no lanzó excepción')
nan = ok_p.copy(); nan[0, 5] = np.nan
inf = ok_p.copy(); inf[1, 9] = np.inf
need_ecg = 'ecg' in bundle['inputs']; need_ppg = 'ppg' in bundle['inputs']
good = lambda m: (ok_p if m == 'ppg' else ok_e) if m in bundle['inputs'] else None
m0 = bundle['inputs'][0]
for label, bad in [('NaN', nan), ('Inf', inf), ('forma incorrecta', ok_p[:, :3000]), ('segmento plano', np.ones_like(ok_p)), ('dtype entero', ok_p.astype(np.int32))]:
    kw = {m: good(m) for m in ('ppg', 'ecg')}
    kw[m0] = bad
    # el canal a corromper debe ser uno que el modelo use
    must_fail(label, ppg_signal=kw['ppg'], ecg_signal=kw['ecg'])
if need_ecg and need_ppg:
    must_fail('falta ECG', ppg_signal=ok_p)
print('una sola señal 1-D (3750,) ->', predict_raw(ok_p[0] if need_ppg else None, ok_e[0] if need_ecg else None, bundle=bundle).round(1))

OK  NaN: ppg: contiene NaN o Inf
OK  Inf: ppg: contiene NaN o Inf
OK  forma incorrecta: ppg: forma (2, 3000), se esperaba (n,3750) o (3750,)
OK  segmento plano: ppg: segmento plano (std <= 1e-06)
OK  dtype entero: ppg: dtype int32 no es float


una sola señal 1-D (3750,) -> [[101.   46.7]]
